# ToyAIKit tools

Goal: swap the hand-written tool-call loop from `02-agent-basics` for toyaikit, so the tool
registry, the loop and the chat display come from a library instead of our own code.

## Load the API key

Walks up from this folder to the `.env` at the repo root, so it works at any depth.

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Install toyaikit

Already in `pyproject.toml`, so this just confirms it - the output shows nothing new to install.
After a real first install, restart the kernel or the import further down won't find it.

In [ ]:
!uv add toyaikit

# Tools for the agent

Goal: the same docs index as before, plus two tools - one that reads from it, one that writes to it.

## Build the search index

`AppendableIndex` rather than plain `Index`, because `add_entry` below adds to it after `fit`.

In [ ]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

## Search tool

The function the agent can call, plus the schema that tells the model what argument it takes.

In [ ]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

## Add-entry tool

A write tool, so the model has a second tool to choose between. None of the questions below ask
for it, so expect only `search` calls.

In [ ]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

## Agent instructions

Asks for one search, then two more based on what came back, then an answer - and an explanation
before each search, so the output shows why it picked each query.

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

# Running the agent with toyaikit

Goal: replace our own while-loop with toyaikit's runner and get the same answer back.

## Register the tools

`Tools` keeps each function next to its schema, so the runner can send the schemas to the model
and call the right function when the model asks for it.

In [ ]:
from toyaikit.tools import Tools

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

In [ ]:
agent_tools.get_tools()

`get_tools()` gives back exactly the schemas we wrote - toyaikit collects them, it doesn't
rewrite them.

## Wrap the OpenAI client

`gpt-4o-mini` keeps every run here near zero cost.

In [ ]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

## Create the runner

`chat_interface` is left out for now. `loop()` doesn't need one - only the interactive chat at
the end does.

In [ ]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

## Ask one question

`loop()` runs the whole search-and-answer cycle and only returns once the model stops calling
tools. Nothing prints - everything comes back in `result`.

In [ ]:
result = agent.loop('how can I create evidently dahsbord?')

## Read the answer and the cost

In [ ]:
print(result.last_message)

In [ ]:
result.cost.total_cost

The wording of the answer changes from run to run; the cost for this question lands around
$0.003.

# Watching the agent work

Goal: see each tool call and reply as it happens, not just the final message.

## Set up the display

`IPythonChatInterface` renders HTML in the notebook, and `DisplayingRunnerCallback` hooks it into
the loop so every step gets shown.

In [ ]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [ ]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface) 

## Same question, now visible

In [ ]:
result = agent.loop(
    'how can I create evidently dahsbord?',
    callback=runner_callback
)

Three `Response received` rounds, in line with the three iterations the instructions ask for.
The HTML blocks are the search calls and the final answer.

## Follow-up question

`previous_messages` passes the first conversation back in, which is why "show me the code" works
without saying code for what. It goes into `result2`, so `result` still holds the first run.

In [ ]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

# Interactive chat

Goal: set up the runner for a turn-by-turn chat.

## Rebuild the runner with a chat interface

`run()` reads your input through the chat interface, so the runner needs one. This notebook stops
before calling `run()` - `toyaikit-tools.ipynb` has the chat itself.

In [ ]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

## Check the cost

Not run yet. Nothing after the visible run reassigns `result`, so this reads that run's cost,
not anything from the chat setup above.

In [ ]:
result.cost.total_cost